<a href="https://colab.research.google.com/github/zia207/Causal_Inference_R/blob/main/Notebook/02_08_05_01_05_multi_arm_causal_forest_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![R Banner](https://drive.google.com/uc?id=13Qd1q78-lCDh7Yp9jwWqvdPYE-vUAXtt)

# 5.1.5 Multi-arm/multi-outcome Causal Forest

This section introduces the **multi-arm/multi-outcome causal forest**, an extension of the standard Causal Forest that can handle multiple treatment arms and multiple outcomes simultaneously. This method is particularly useful in complex experimental settings where researchers want to understand how different treatments affect various outcomes across heterogeneous populations.

## Overview

A **Multi-Arm/Multi-Outcome Causal Forest** extends the standard Causal Forest to handle two common real-world complications simultaneously: there may be more than two treatment options (multiple arms), and the researcher may care about more than one outcome at the same time (multiple outcomes). It remains fully within the Generalized Random Forest (GRF) framework, preserving the core properties of honest estimation and non-parametric flexibility.

The key promise of this extension is **information sharing**: by modeling treatment arms and outcomes jointly, the forest can borrow strength across related estimates rather than fitting each comparison in isolation — improving precision, especially when sample sizes per arm are limited.

### When to Use It

A multi-arm/multi-outcome causal forest is the right tool when the research question is of the form: *"How does each treatment affect each outcome, and does this vary across subgroups?"*

Concrete settings include:

-   **Clinical trials** comparing several drugs or dosing regimens on multiple patient health metrics simultaneously (e.g., blood pressure, cholesterol, adverse events).
-   **Policy evaluation** where several interventions (e.g., cash transfers, job training, mentoring) are tested and success is measured across multiple dimensions (e.g., employment rate, income, school attendance).
-   **Marketing experiments** with multiple campaign strategies evaluated on several customer metrics (e.g., click-through rate, conversion, retention).

In all these cases, fitting separate single-arm, single-outcome models would ignore correlations between outcomes and inflate estimation variance — particularly when some arms have few observations.

### Key Concepts

**Multi-arm treatment effects.** With $J$ treatment arms (including control $W = 0$), the forest estimates $J$ separate conditional effects for each individual:

$$\tau_j(X) = E[Y \mid W = j,\, X = x] - E[Y \mid W = 0,\, X = x], \quad j = 1, \ldots, J$$

**Multi-outcome treatment effects.** With $K$ outcomes $Y_1, \ldots, Y_K$, the forest estimates an effect for each combination of arm $j$ and outcome $k$:

$$\tau_{j,k}(X) = E[Y_k \mid W = j,\, X = x] - E[Y_k \mid W = 0,\, X = x]$$

The full output for an individual is therefore a $J \times K$ matrix of treatment effect estimates.

**Joint splitting criterion.** Unlike fitting $J \times K$ independent forests, a single tree's splits are chosen to maximize heterogeneity across *all* arms and outcomes simultaneously, using a joint loss function that respects the covariance structure among outcomes. This is what enables information sharing.

**Nuisance estimation.** Propensity scores $\hat{e}_j(X) = P(W = j \mid X)$ and outcome regressions $\hat{\mu}_k(X)$ are estimated using separate regression forests. Residualizing with these nuisances — forming $\tilde{Y}_k = Y_k - \hat{\mu}_k(X)$ and $\tilde{W}_j = \mathbf{1}(W=j) - \hat{e}_j(X)$ — removes confounding before the causal forest estimates effects.

**Honest estimation.** As in all GRF methods, each tree uses a separate subsample to find splits ($S_1$) and to estimate effects within leaves ($S_2$), preventing overfitting and ensuring asymptotically valid confidence intervals.

### How It Works

A few things worth emphasizing from the pipeline:

**The** $J \times K$ matrix in step 5 is the defining output. Each row is a treatment arm; each column is an outcome. For an individual with covariates $X = x$, every cell gives a separate CATE estimate with its own confidence interval. This is what distinguishes the multi-arm/multi-outcome forest from running $J \times K$ separate causal forests — the joint splitting criterion in step 3 means the covariate partitions reflect *shared* structure across all cells, not independent ones.

**Step 2 (nuisance estimation) is more complex here** than in the binary treatment case. With $J$ arms, the forest must estimate $J$ propensity scores $\hat{e}_j(X)$ — one per arm — and $K$ outcome regressions $\hat{\mu}_k(X)$. These are estimated via separate regression forests and used to form the residuals $\tilde{W}_j$ and $\tilde{Y}_k$ that the causal forest then splits on.

**Information sharing is the key efficiency gain.** When outcomes $Y_1$ and $Y_2$ are correlated (e.g., systolic and diastolic blood pressure), the joint covariance structure in the splitting criterion means that a split informative for one outcome also informs the other. This can substantially reduce variance compared to independent forests, particularly in small-sample regimes.

### Key Differences from Standard Causal Forest

-   `Standard Causal Forest`: Handles one treatment (binary or continuous) and one outcome, estimating a single treatment effect per individual.
-   `Multi-Arm/Multi-Outcome Causal Forest`:
    -   Models multiple treatments (e.g., multiple drugs) and/or multiple outcomes (e.g., multiple health metrics).
    -   Shares information across arms/outcomes to improve estimation efficiency.
    -   Uses a joint objective function to optimize splits for all treatments and outcomes simultaneously.

### Advantages

-   Handles complex experiments with multiple treatments and outcomes.
-   Captures heterogeneity in treatment effects across subgroups.
-   Robust to non-linear relationships and high-dimensional covariates.
-   Improves efficiency by sharing information across arms/outcomes.

### Limitations

-   Requires sufficient sample size for each treatment arm and outcome.
-   Assumes unconfoundedness (no unmeasured confounders) unless combined with other methods (e.g., instrumental variables).
-   Computationally intensive for large datasets or many outcomes.
-   Interpretability can be challenging due to complex output (multiple effects per individual).

## Multi-arm/multi-outcome Causal Forestt in R

This tutorial demonstrates how to implement a **multi-arm/multi-outcome causal forest** in R using the `{grf}` package. The example uses the `lung` dataset from the `{survival}` package, simulating a multi-arm treatment scenario with multiple outcomes.

## Setup R in Python Runtime - Install {rpy2}

{rpy2} allows running R code in Colab Python runtime via `%%R` magic.

In [ ]:
!pip uninstall rpy2 -y
!pip install rpy2==3.5.1
%load_ext rpy2.ipython

## Mount Google Drive

Mount Google Drive if your data or R library folder is stored there.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Set Up

### Check and Install Required R Packages

Following R packages are required to run this notebook. If any of these packages are not installed, you can install them using the code below:

`tidyverse`, `plyr`, `RCausalML`, `causaldata`, `mlbench`, `survival`, `BART`, `grf`, `kernelshap`, `shapviz`

In [ ]:
%%R
packages <- c(
  "tidyverse",
  "plyr",
  "RCausalML",
  "causaldata",
  "mlbench",
  "survival",
  "BART",
  "grf",
  "kernelshap",
  "shapviz"
)

### Install Missing Packages

In [ ]:
%%R
# Install missing packages
new.packages <- packages[!(packages %in% installed.packages(lib='drive/My Drive/R/')[,"Package"])]
if(length(new.packages)) install.packages(new.packages, lib='drive/My Drive/R/')

### Verify Installation

In [ ]:
%%R
# Verify installation
cat("Installed packages:\n")
print(sapply(packages, requireNamespace, quietly = TRUE))

### Load R Packages

In [ ]:
%%R
# set library path
.libPaths('drive/My Drive/R')
# Verify installation
cat("Installed packages:\n")
print(sapply(packages, requireNamespace, quietly = TRUE))

### Check Loaded Packages

In [ ]:
%%R
# Check loaded packages
cat("Successfully loaded packages:\n")

In [ ]:
%%R
# Load packages with suppressed messages
invisible(lapply(packages, function(pkg) {
  suppressPackageStartupMessages(library(pkg, character.only = TRUE))
}))
# Check loaded packages
cat("Successfully loaded packages:\n")
print(search()[grepl("package:", search())])

### Load and Prepare the Lung Dataset

The lung dataset includes survival data for lung cancer patients. Since lung lacks a treatment variable and multiple outcomes, we:

-   Simulates a multi-arm `treatment` variable with three levels: `placebo`, `A`, and `B` and Assigns probabilities (40% placebo, 30% A, 30% B) for each treatment.

-   Simulates a binary secondary outcome (e.g., `health_status`, 1 = yes, 0 = no) with treatment-dependent probabilities.

-   Use log-transformed survival time (`log_time`) as the primary outcome to ensure numerical stability

In [ ]:
%%R
# Load and modify the lung data
data(lung, package = "survival")
lung_data <- lung %>%
  dplyr::filter(!is.na(age), !is.na(sex), !is.na(ph.ecog)) %>%
  dplyr::mutate(
    treatment = as.factor(sample(c("placebo", "A", "B"), n(), replace = TRUE, prob = c(0.4, 0.3, 0.3))),
    health_status = rbinom(n(), 1, prob = 0.3 + 0.1 * (treatment == "A") + 0.15 * (treatment == "B")),
    log_time = log(time)
  )

# Define covariates, treatment, and outcomes
X <- lung_data %>% select(age, sex, ph.ecog) %>% as.matrix()
W <- lung_data$treatment
Y <- lung_data %>% select(log_time, health_status) %>% as.matrix()
colnames(Y) <- c("log_time", "health_status")

# Split into training and test sets
set.seed(123)
n <- nrow(lung_data)
train_idx <- sample(1:n, size = 0.8 * n)
test_idx <- setdiff(1:n, train_idx)
X_train <- X[train_idx, ]
X_test <- X[test_idx, ]
W_train <- W[train_idx]
W_test <- W[test_idx]
Y_train <- Y[train_idx, ]
Y_test <- Y[test_idx, ]

### Train the Multi-arm/Multi-outcome Causal Forest

Since we have multiple treatment arms ("placebo", "A", "B") and multiple outcomes (log_time and health_status), and the grf package’s `multi_arm_causal_forest` function is designed for multiple treatments but a single outcome, we fit a separate model for each outcome.

In [ ]:
%%R
macf_log_time <- multi_arm_causal_forest(X = X_train,
                                         Y = Y_train[, "log_time"],
                                         W = W_train)
macf_health_status <- multi_arm_causal_forest(X = X_train,
                                              Y = Y_train[, "health_status"],
                                              W = W_train)

### Predict on the test data set

In [ ]:
%%R
# Predict treatment effects
# Predict treatment effects on the test set
tau_hat_log_time <- predict(macf_log_time, newdata = X_test)
tau_hat_health_status <- predict(macf_health_status, newdata = X_test)

### Calculate Doubly Robust Average Treatment Effects (AIPW)

We compute the average treatment effects (ATE) for each arm relative to "placebo" using the doubly robust Augmented Inverse Propensity Weighting (AIPW) method provided by {grf}.

In [ ]:
%%R
# Calculate ATE for log_time
ate_log_time <- average_treatment_effect(macf_log_time)

# Calculate ATE for health_status
ate_health_status <- average_treatment_effect(macf_health_status)

# Display the results
print("Average Treatment Effects for log_time:")
print(ate_log_time)
print("Average Treatment Effects for health_status:")
print(ate_health_status)

### Plot Variable Importance

Finally, we plot the importance of each covariate (age, sex, ph.ecog) for each outcome using ggplot2.

#### log_time Outcome

In [ ]:
%%R
# Variable importance for log_time
var_imp_log_time <- variable_importance(macf_log_time)
var_imp_df_log_time <- data.frame(
  Variable = colnames(X),
  Importance = var_imp_log_time
)

ggplot(var_imp_df_log_time, aes(x = reorder(Variable, Importance), y = Importance)) +
  geom_bar(stat = "identity") +
  coord_flip() +
  labs(title = "Variable Importance for log_time", x = "Variable", y = "Importance") +
  theme_minimal()

#### Health_status Outcome

In [ ]:
%%R
# Variable importance for health_status
var_imp_health_status <- variable_importance(macf_health_status)
var_imp_df_health_status <- data.frame(
  Variable = colnames(X),
  Importance = var_imp_health_status
)

ggplot(var_imp_df_health_status, aes(x = reorder(Variable, Importance), y = Importance)) +
  geom_bar(stat = "identity") +
  coord_flip() +
  labs(title = "Variable Importance for health_status", x = "Variable", y = "Importance") +
  theme_minimal()

### Compute Kernel SHAP values for CATE using shapviz

We use the `explain_cate()` function from the RCausalML package (see `R/shapviz_integration.R`) to compute SHAP values for the CATE predictions of each outcome. Here we show both `log_time` and `health_status` outcomes.

#### Step  “placebo” the baseline (highly recommended for interpretability)
Run this **before** refitting the forests (or right after you create `W_train`):

In [ ]:
%%R
# Re-level so placebo is the reference (baseline)
W_train <- relevel(as.factor(W_train), ref = "placebo")
W_test  <- relevel(as.factor(W_test),  ref = "placebo")   # optional but consistent

# Refit both forests with the new baseline
macf_log_time <- multi_arm_causal_forest(
  X = X_train,
  Y = Y_train[, "log_time"],
  W = W_train
)

macf_health_status <- multi_arm_causal_forest(
  X = X_train,
  Y = Y_train[, "health_status"],
  W = W_train
)

After this, the column names in `predict()$predictions` will be **"A - placebo"** and **"B - placebo"** (exactly what we want).

#### Check the new structure (quick diagnostic)

In [ ]:
%%R
p <- predict(macf_log_time, newdata = head(X_test, 5))
print(colnames(p$predictions[, , 1]))

#### Define clean prediction wrappers (one per contrast)

Add these functions right before the SHAP computation:

In [ ]:
%%R
# Prediction wrapper for log_time outcome
pred_A_vs_placebo_log <- function(object, newdata) {
  p <- predict(object, newdata = newdata)
  p$predictions[, "A - placebo", 1]          # extracts the scalar vector we need
}

pred_B_vs_placebo_log <- function(object, newdata) {
  p <- predict(object, newdata = newdata)
  p$predictions[, "B - placebo", 1]
}

# Same wrappers for health_status outcome
pred_A_vs_placebo_health <- function(object, newdata) {
  p <- predict(object, newdata = newdata)
  p$predictions[, "A - placebo", 1]
}

pred_B_vs_placebo_health <- function(object, newdata) {
  p <- predict(object, newdata = newdata)
  p$predictions[, "B - placebo", 1]
}

#### Compute Kernel SHAP (the real replacement for `explain_cate()`)

In [ ]:
%%R
library(kernelshap)
library(shapviz)
library(future)

#  Parallel backend
n_cores <- max(1L, parallel::detectCores() - 1L)
plan(multisession, workers = n_cores)

# Small background sample (50 rows) – makes Kernel SHAP fast and stable
set.seed(42)
bg_idx <- sample(nrow(X), min(50, nrow(X)))
bg_small <- X[bg_idx, , drop = FALSE]

# ── log_time outcome ───────────────────────────────────────────────
shap_A_log <- kernelshap(
  object   = macf_log_time,
  X        = X_test,                    # or X[1:500, ] if you want faster
  bg_X     = bg_small,
  pred_fun = pred_A_vs_placebo_log,
  verbose  = FALSE
)

shap_B_log <- kernelshap(
  object   = macf_log_time,
  X        = X_test,
  bg_X     = bg_small,
  pred_fun = pred_B_vs_placebo_log,
  verbose  = FALSE
)

# ── health_status outcome ──────────────────────────────────────────
shap_A_health <- kernelshap(
  object   = macf_health_status,
  X        = X_test,
  bg_X     = bg_small,
  pred_fun = pred_A_vs_placebo_health,
  verbose  = FALSE
)

shap_B_health <- kernelshap(
  object   = macf_health_status,
  X        = X_test,
  bg_X     = bg_small,
  pred_fun = pred_B_vs_placebo_health,
  verbose  = FALSE
)

# Reset workers
plan(sequential)

### Exact Kernel SHAP values (preview)

In [ ]:
%%R
library(knitr)

# Show a small preview so values appear in HTML
sv_A_log <- shapviz(shap_A_log)
cat("**SHAP matrix (log_time: A vs placebo)**\n\n")
print(knitr::kable(head(as.data.frame(sv_A_log$S), 10), digits = 4))

### Turn into shapviz objects and visualize (panel display)

In [ ]:
%%R
# Convert to shapviz objects
shp_A_log    <- shapviz(shap_A_log)
shp_B_log    <- shapviz(shap_B_log)
shp_A_health <- shapviz(shap_A_health)
shp_B_health <- shapviz(shap_B_health)

### Importance Plots

In [ ]:
%%R
library(patchwork) # For panel display

# Importance plots (beeswarm + bar) for each (treatment, outcome)
plt1 <- sv_importance(shp_A_log,    kind = "both") + ggtitle("log_time: A vs placebo")
plt2 <- sv_importance(shp_B_log,    kind = "both") + ggtitle("log_time: B vs placebo")
plt3 <- sv_importance(shp_A_health, kind = "both") + ggtitle("health_status: A vs placebo")
plt4 <- sv_importance(shp_B_health, kind = "both") + ggtitle("health_status: B vs placebo")

# Arrange plots in a 2x2 panel
(plt1 | plt2) / (plt3 | plt4)

### Dependence plots for each covariate

In [ ]:
%%R
# List available feature names for each SHAP object
print("Available features in SHAP object (log_time):")
print(colnames(shp_A_log$X))
print("Available features in SHAP object (health_status):")
print(colnames(shp_A_health$X))

# For illustration, show dependence plots for the first three available features for BOTH log_time and health_status
available_feats_log <- colnames(shp_A_log$X)
available_feats_health <- colnames(shp_A_health$X)

# Logging available features for each outcome
print("Features (log_time):")
print(available_feats_log)
print("Features (health_status):")
print(available_feats_health)

In [ ]:
%%R
# Dependence plots for log_time outcome (A vs placebo)
plt_A_log1 <- sv_dependence(shp_A_log, v = available_feats_log[1]) + ggtitle(paste("log_time (A):", available_feats_log[1]))
plt_A_log2 <- sv_dependence(shp_A_log, v = available_feats_log[2]) + ggtitle(paste("log_time (A):", available_feats_log[2]))
plt_A_log3 <- sv_dependence(shp_A_log, v = available_feats_log[3]) + ggtitle(paste("log_time (A):", available_feats_log[3]))

# Dependence plots for health_status outcome (A vs placebo)
plt_A_health1 <- sv_dependence(shp_A_health, v = available_feats_health[1]) + ggtitle(paste("health_status (A):", available_feats_health[1]))
plt_A_health2 <- sv_dependence(shp_A_health, v = available_feats_health[2]) + ggtitle(paste("health_status (A):", available_feats_health[2]))
plt_A_health3 <- sv_dependence(shp_A_health, v = available_feats_health[3]) + ggtitle(paste("health_status (A):", available_feats_health[3]))

library(patchwork)
# Display all dependence plots in a 2x3 grid (top: log_time, bottom: health_status)
(plt_A_log1 | plt_A_log2 | plt_A_log3) /
(plt_A_health1 | plt_A_health2 | plt_A_health3)

## Summary and Conclusion

Multi-arm/multi-outcome causal forests are powerful tools for estimating heterogeneous treatment effects in complex settings with multiple treatments and outcomes. By leveraging the flexibility of random forests, they provide robust estimates that account for individual differences and correlations between outcomes. This approach is particularly useful in fields like healthcare, policy evaluation, and marketing, where understanding the nuanced effects of different interventions is crucial. This tutorial demonstrated how to implement a multi-arm/multi-outcome causal forest in R, using the `{RCausalML}` package intregated with with {grf} packages to analyze treatment effects on multiple outcomes simultaneously. This approach allows researchers to gain deeper insights into the effectiveness of various interventions across diverse populations and outcomes, ultimately leading to more informed decision-making.

We also integrated SHAP values using the `kernelshap` and `shapviz` packages to interpret the heterogeneous treatment effects, providing a comprehensive framework for both estimation and interpretation in multi-arm/multi-outcome causal inference.

If you want a boosted (XGBoost-based) multi-arm approach that fits one outcome model per arm and then forms contrasts relative to a baseline, see `08-multi-arm-causal-boost-r.qmd`.

## Resources

1.  Wager, S., & Athey, S. (2018). Estimation and Inference of Heterogeneous Treatment Effects using Random Forests. JASA.

2.  Athey, S., Tibshirani, J., & Wager, S. (2019). Generalized Random Forests. Ann. Statist.

3.  Multi-arm/multi-outcome implementation available in grf R package